In [1]:
import numpy as np
import pandas as pd
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_q

In [2]:
def create_simulated_dataset(p_vector, p_constant=False, N=7, n_c=20, N_0=1e6, R=5e7, p=0.6, k=1e11, amp_rounds=25, seed=0):
    
    np.random.seed(seed)
    
    lengths = [torch.arange(n_c, dtype=torch.int8) for i in range(N)]
    all_seq = torch.cartesian_prod(*lengths)
    all_seq = all_seq.long()
    
    #first poisson sampling
    C = np.random.poisson(N_0*p_vector)
    
    #adjust dataset
    all_seq = all_seq[C > 0]
    C = C[C > 0]

    #binomial process
    if p_constant:
        for _ in range(amp_rounds):
            C += np.random.binomial(n=C, p=p)
    else:
        p_variable=[]
        C_tot=[]
        
        for _ in range(amp_rounds):
            p = 1. / (1 + C.sum()/k)
            C_tot.append(C.sum())
            p_variable.append(p)
            C += np.random.binomial(n=C, p=p)
            print('%.2e'%C.sum())
        
    #second poisson sampling
    C = np.random.poisson(R*C/C.sum()) 
        
    #adjust dataset
    all_seq = all_seq[C > 0]
    C = C[C > 0]
    
    if p_constant:
        return all_seq.numpy()
    
    return all_seq.numpy(), C, C_tot, p_variable

In [3]:
def infer_fields(data, counts):
    
    fields = np.zeros([7,20])
    for pos in range(7):
        for color in range(20):
            fields[pos,color] = (counts*((data[:,pos] == color).astype('int'))).sum() 

    print(counts.sum(), fields.sum(1))

    fields = fields[:,:] / fields[:,0][:,np.newaxis]
    return torch.tensor(np.log(fields))

def log10p_vector_from_fields(fields):
    
    lengths = [torch.arange(20, dtype=torch.int8) for i in range(7)]
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(20**7,dtype=torch.float32)

    for i in range(7):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    
    p_vector = torch.exp(p_vector)
    
    Z = torch.exp(fields).sum(1).prod()
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector) / np.log(10)

def log10q_vector_func(data, fields):
    
    # generate q_vector
    q_vector = torch.zeros(len(data))

    for i in range(7):
        q_vector += fields[i,data[:,i]]
        print(i)

    q_vector = torch.exp(q_vector)

    Z = torch.exp(fields).sum(1).prod()

    q_vector /= Z

    print(q_vector.sum())
    return np.log10(q_vector.numpy())

In [4]:
data = pd.read_csv('../data/Byrne.csv',index_col=0).query('T0 > 0')
data = data.sort_values('T0', ascending=False)
data = data.iloc[1:,:]

counts = data['T0'].to_numpy()
data = data.iloc[:,:7].to_numpy()

In [5]:
fields = infer_fields(data, counts)

12195473 [12195473. 12195473. 12195473. 12195473. 12195473. 12195473. 12195473.]


In [6]:
data_log10p_vector = log10p_vector_from_fields(fields)

0
1
2
3
4
5
6
tensor(1.)


In [7]:
R = counts.sum()
F = np.loadtxt('../binning_real_data_q/F_Byrne.csv')[0]
N_0 = int(R / (F-1))
F, N_0

(8.922473368689328, 1539351)

## different amp_rounds 

In [12]:
k = 1e5 * N_0

for amp_rounds in range(0,30,5):

    print('amplification_rounds = %d'%amp_rounds)
    
    data, counts, C_tot, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0, k=k, amp_rounds=amp_rounds)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_%damp_rounds.csv'%amp_rounds)
    
    del(log10p_vector)
    del(sorted_log10p_vector)

amplification_rounds = 0
12202649 [12202649. 12202649. 12202649. 12202649. 12202649. 12202649. 12202649.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0065)
0
tensor(1279999996) tensor(1279622767)
elements in the bin: 377229
nonzeros: 19192
1
tensor(1279622767) tensor(1279055760)
elements in the bin: 567007
nonzeros: 19192
2
tensor(1279055760) tensor(1278361488)
elements in the bin: 694272
nonzeros: 19192
3
tensor(1278361488) tensor(1277550945)
elements in the bin: 810543
nonzeros: 19192
4
tensor(1277550945) tensor(1276641191)
elements in the bin: 909754
nonzeros: 19192
5
tensor(1276641191) tensor(1275619059)
elements in the bin: 1022132
nonzeros: 19192
6
tensor(1275619059) tensor(1274504088)
elements in the bin: 1114971
nonzeros: 19192
7
tensor(1274504088) tensor(1273300941)
elements in the bin: 1203147
nonzeros: 19192
8
tensor(1273300941) tensor(1272003701)
elements in the bin: 1297240
nonzeros: 19192
9
tensor(1272003701) tensor(1270624203)
elements in the bin: 1379498
nonzeros: 1

35
tensor(1201174872) tensor(1196650523)
elements in the bin: 4524349
nonzeros: 19192
36
tensor(1196650523) tensor(1192006924)
elements in the bin: 4643599
nonzeros: 19192
37
tensor(1192006924) tensor(1187117034)
elements in the bin: 4889890
nonzeros: 19192
38
tensor(1187117034) tensor(1181953116)
elements in the bin: 5163918
nonzeros: 19192
39
tensor(1181953116) tensor(1176616910)
elements in the bin: 5336206
nonzeros: 19192
40
tensor(1176616910) tensor(1171039478)
elements in the bin: 5577432
nonzeros: 19192
41
tensor(1171039478) tensor(1165297794)
elements in the bin: 5741684
nonzeros: 19192
42
tensor(1165297794) tensor(1159283643)
elements in the bin: 6014151
nonzeros: 19192
43
tensor(1159283643) tensor(1153016393)
elements in the bin: 6267250
nonzeros: 19192
44
tensor(1153016393) tensor(1146549857)
elements in the bin: 6466536
nonzeros: 19192
45
tensor(1146549857) tensor(1139737569)
elements in the bin: 6812288
nonzeros: 19192
46
tensor(1139737569) tensor(1132668178)
elements in t

53
tensor(1083199539) tensor(1073628944)
elements in the bin: 9570595
nonzeros: 19192
54
tensor(1073628944) tensor(1063589748)
elements in the bin: 10039196
nonzeros: 19192
55
tensor(1063589748) tensor(1053176713)
elements in the bin: 10413035
nonzeros: 19192
56
tensor(1053176713) tensor(1041970311)
elements in the bin: 11206402
nonzeros: 19192
57
tensor(1041970311) tensor(1030282511)
elements in the bin: 11687800
nonzeros: 19192
58
tensor(1030282511) tensor(1018101924)
elements in the bin: 12180587
nonzeros: 19192
59
tensor(1018101924) tensor(1005023197)
elements in the bin: 13078727
nonzeros: 19192
60
tensor(1005023197) tensor(991209165)
elements in the bin: 13814032
nonzeros: 19192
61
tensor(991209165) tensor(976654587)
elements in the bin: 14554578
nonzeros: 19192
62
tensor(976654587) tensor(961507862)
elements in the bin: 15146725
nonzeros: 19192
63
tensor(961507862) tensor(945406758)
elements in the bin: 16101104
nonzeros: 19192
64
tensor(945406758) tensor(928023197)
elements in 

67
tensor(889609283) tensor(868503702)
elements in the bin: 21105581
nonzeros: 19192
68
tensor(868503702) tensor(844955561)
elements in the bin: 23548141
nonzeros: 19192
69
tensor(844955561) tensor(819714986)
elements in the bin: 25240575
nonzeros: 19192
70
tensor(819714986) tensor(792197965)
elements in the bin: 27517021
nonzeros: 19192
71
tensor(792197965) tensor(761870713)
elements in the bin: 30327252
nonzeros: 19192
72
tensor(761870713) tensor(728114660)
elements in the bin: 33756053
nonzeros: 19192
73
tensor(728114660) tensor(689762358)
elements in the bin: 38352302
nonzeros: 19192
74
tensor(689762358) tensor(646228437)
elements in the bin: 43533921
nonzeros: 19192
75
tensor(646228437) tensor(594386057)
elements in the bin: 51842380
nonzeros: 19192
76
tensor(594386057) tensor(531697448)
elements in the bin: 62688609
nonzeros: 19192
77
tensor(531697448) tensor(450972897)
elements in the bin: 80724551
nonzeros: 19192
78
tensor(450972897) tensor(334075102)
elements in the bin: 11689

79
tensor(334333243) tensor(1079574)
elements in the bin: 333253669
nonzeros: 19253
amplification_rounds = 25
3.08e+06
6.16e+06
1.23e+07
2.47e+07
4.93e+07
9.86e+07
1.97e+08
3.94e+08
7.87e+08
1.57e+09
3.12e+09
6.19e+09
1.21e+10
2.34e+10
4.37e+10
7.77e+10
1.29e+11
2.00e+11
2.87e+11
3.87e+11
4.97e+11
6.14e+11
7.37e+11
8.65e+11
9.95e+11
12192320 [12192320. 12192320. 12192320. 12192320. 12192320. 12192320. 12192320.]
0
1
2
3
4
5
6
tensor(1.0000)
0
1
2
3
4
5
6
tensor(0.0065)
0
tensor(1279999996) tensor(1279623115)
elements in the bin: 376881
nonzeros: 19192
1
tensor(1279623115) tensor(1279056070)
elements in the bin: 567045
nonzeros: 19192
2
tensor(1279056070) tensor(1278362446)
elements in the bin: 693624
nonzeros: 19192
3
tensor(1278362446) tensor(1277551495)
elements in the bin: 810951
nonzeros: 19192
4
tensor(1277551495) tensor(1276639416)
elements in the bin: 912079
nonzeros: 19192
5
tensor(1276639416) tensor(1275618177)
elements in the bin: 1021239
nonzeros: 19192
6
tensor(1275618177) 

In [13]:
np.save('p_variable_%.1ek.npy'%k,p_variable)
np.save('C_tot_%.1ek.npy'%k, C_tot)

## different k 

In [14]:
for k in N_0*np.array([1e1, 1e3, 1e7, 1e9]):
    
    data, counts, C_tot, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0, k=k)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_%.1ek.csv'%k)
    
    del(log10p_vector)
    del(sorted_log10p_vector)
    
    np.save('p_variable_%.1ek.npy'%k,p_variable)
    np.save('C_tot_%.1ek.npy'%k,C_tot)

2.94e+06
5.41e+06
9.41e+06
1.53e+07
2.29e+07
3.21e+07
4.25e+07
5.38e+07
6.58e+07
7.83e+07
9.11e+07
1.04e+08
1.18e+08
1.31e+08
1.45e+08
1.59e+08
1.73e+08
1.87e+08
2.01e+08
2.16e+08
2.30e+08
2.45e+08
2.59e+08
2.74e+08
2.88e+08
12197543 [12197543. 12197543. 12197543. 12197543. 12197543. 12197543. 12197543.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0064)
0
tensor(1279999996) tensor(1279622620)
elements in the bin: 377376
nonzeros: 19036
1
tensor(1279622620) tensor(1279055169)
elements in the bin: 567451
nonzeros: 19036
2
tensor(1279055169) tensor(1278362007)
elements in the bin: 693162
nonzeros: 19036
3
tensor(1278362007) tensor(1277548596)
elements in the bin: 813411
nonzeros: 19036
4
tensor(1277548596) tensor(1276638956)
elements in the bin: 909640
nonzeros: 19036
5
tensor(1276638956) tensor(1275619091)
elements in the bin: 1019865
nonzeros: 19036
6
tensor(1275619091) tensor(1274504288)
elements in the bin: 1114803
nonzeros: 19036
7
tensor(1274504288) tensor(1273304176)
elements i

tensor(1196651856) tensor(1192015729)
elements in the bin: 4636127
nonzeros: 19191
37
tensor(1192015729) tensor(1187110115)
elements in the bin: 4905614
nonzeros: 19191
38
tensor(1187110115) tensor(1181966790)
elements in the bin: 5143325
nonzeros: 19191
39
tensor(1181966790) tensor(1176614115)
elements in the bin: 5352675
nonzeros: 19191
40
tensor(1176614115) tensor(1171057077)
elements in the bin: 5557038
nonzeros: 19191
41
tensor(1171057077) tensor(1165306739)
elements in the bin: 5750338
nonzeros: 19191
42
tensor(1165306739) tensor(1159297088)
elements in the bin: 6009651
nonzeros: 19191
43
tensor(1159297088) tensor(1153021626)
elements in the bin: 6275462
nonzeros: 19191
44
tensor(1153021626) tensor(1146547438)
elements in the bin: 6474188
nonzeros: 19191
45
tensor(1146547438) tensor(1139734739)
elements in the bin: 6812699
nonzeros: 19191
46
tensor(1139734739) tensor(1132673534)
elements in the bin: 7061205
nonzeros: 19191
47
tensor(1132673534) tensor(1125313663)
elements in the 

52
tensor(1092459086) tensor(1083177125)
elements in the bin: 9281961
nonzeros: 19192
53
tensor(1083177125) tensor(1073651840)
elements in the bin: 9525285
nonzeros: 19192
54
tensor(1073651840) tensor(1063597426)
elements in the bin: 10054414
nonzeros: 19192
55
tensor(1063597426) tensor(1053124431)
elements in the bin: 10472995
nonzeros: 19192
56
tensor(1053124431) tensor(1041974537)
elements in the bin: 11149894
nonzeros: 19192
57
tensor(1041974537) tensor(1030291654)
elements in the bin: 11682883
nonzeros: 19192
58
tensor(1030291654) tensor(1018086411)
elements in the bin: 12205243
nonzeros: 19192
59
tensor(1018086411) tensor(1005018702)
elements in the bin: 13067709
nonzeros: 19192
60
tensor(1005018702) tensor(991205622)
elements in the bin: 13813080
nonzeros: 19192
61
tensor(991205622) tensor(976655265)
elements in the bin: 14550357
nonzeros: 19192
62
tensor(976655265) tensor(961506923)
elements in the bin: 15148342
nonzeros: 19192
63
tensor(961506923) tensor(945406713)
elements in

65
tensor(927927978) tensor(909321489)
elements in the bin: 18606489
nonzeros: 19193
66
tensor(909321489) tensor(889634918)
elements in the bin: 19686571
nonzeros: 19193
67
tensor(889634918) tensor(868398570)
elements in the bin: 21236348
nonzeros: 19193
68
tensor(868398570) tensor(844871546)
elements in the bin: 23527024
nonzeros: 19193
69
tensor(844871546) tensor(819626970)
elements in the bin: 25244576
nonzeros: 19193
70
tensor(819626970) tensor(792122720)
elements in the bin: 27504250
nonzeros: 19193
71
tensor(792122720) tensor(761784368)
elements in the bin: 30338352
nonzeros: 19193
72
tensor(761784368) tensor(728108285)
elements in the bin: 33676083
nonzeros: 19193
73
tensor(728108285) tensor(689666255)
elements in the bin: 38442030
nonzeros: 19193
74
tensor(689666255) tensor(646025946)
elements in the bin: 43640309
nonzeros: 19193
75
tensor(646025946) tensor(594259940)
elements in the bin: 51766006
nonzeros: 19193
76
tensor(594259940) tensor(531619711)
elements in the bin: 62640

## different R

In [15]:
for t in [1,2,4,8,16,32,64,128,256]:

    print('R = %d'%R)
    
    data, counts, C_tot, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_%dt.csv'%t)
    
    del(log10p_vector)
    del(sorted_log10p_vector)
    
    R = int(R / 2)

R = 12195473
3.08e+06
6.16e+06
1.23e+07
2.47e+07
4.93e+07
9.86e+07
1.97e+08
3.94e+08
7.86e+08
1.57e+09
3.11e+09
6.12e+09
1.19e+10
2.25e+10
4.09e+10
6.99e+10
1.11e+11
1.64e+11
2.26e+11
2.95e+11
3.70e+11
4.48e+11
5.30e+11
6.14e+11
7.00e+11
12193101 [12193101. 12193101. 12193101. 12193101. 12193101. 12193101. 12193101.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0065)
0
tensor(1279999996) tensor(1279622348)
elements in the bin: 377648
nonzeros: 19192
1
tensor(1279622348) tensor(1279055876)
elements in the bin: 566472
nonzeros: 19192
2
tensor(1279055876) tensor(1278362555)
elements in the bin: 693321
nonzeros: 19192
3
tensor(1278362555) tensor(1277550945)
elements in the bin: 811610
nonzeros: 19192
4
tensor(1277550945) tensor(1276640279)
elements in the bin: 910666
nonzeros: 19192
5
tensor(1276640279) tensor(1275618786)
elements in the bin: 1021493
nonzeros: 19192
6
tensor(1275618786) tensor(1274508675)
elements in the bin: 1110111
nonzeros: 19192
7
tensor(1274508675) tensor(127330521

35
tensor(1201180978) tensor(1196636648)
elements in the bin: 4544330
nonzeros: 18829
36
tensor(1196636648) tensor(1192002710)
elements in the bin: 4633938
nonzeros: 18829
37
tensor(1192002710) tensor(1187105623)
elements in the bin: 4897087
nonzeros: 18829
38
tensor(1187105623) tensor(1181955450)
elements in the bin: 5150173
nonzeros: 18829
39
tensor(1181955450) tensor(1176622353)
elements in the bin: 5333097
nonzeros: 18829
40
tensor(1176622353) tensor(1171053526)
elements in the bin: 5568827
nonzeros: 18829
41
tensor(1171053526) tensor(1165288123)
elements in the bin: 5765403
nonzeros: 18829
42
tensor(1165288123) tensor(1159299221)
elements in the bin: 5988902
nonzeros: 18829
43
tensor(1159299221) tensor(1152990078)
elements in the bin: 6309143
nonzeros: 18829
44
tensor(1152990078) tensor(1146517977)
elements in the bin: 6472101
nonzeros: 18829
45
tensor(1146517977) tensor(1139728576)
elements in the bin: 6789401
nonzeros: 18829
46
tensor(1139728576) tensor(1132639128)
elements in t

54
tensor(1073586177) tensor(1063568760)
elements in the bin: 10017417
nonzeros: 16548
55
tensor(1063568760) tensor(1053129861)
elements in the bin: 10438899
nonzeros: 16548
56
tensor(1053129861) tensor(1041922282)
elements in the bin: 11207579
nonzeros: 16548
57
tensor(1041922282) tensor(1030296473)
elements in the bin: 11625809
nonzeros: 16548
58
tensor(1030296473) tensor(1018020022)
elements in the bin: 12276451
nonzeros: 16548
59
tensor(1018020022) tensor(1004869050)
elements in the bin: 13150972
nonzeros: 16548
60
tensor(1004869050) tensor(991090538)
elements in the bin: 13778512
nonzeros: 16548
61
tensor(991090538) tensor(976536662)
elements in the bin: 14553876
nonzeros: 16548
62
tensor(976536662) tensor(961373158)
elements in the bin: 15163504
nonzeros: 16548
63
tensor(961373158) tensor(945350647)
elements in the bin: 16022511
nonzeros: 16548
64
tensor(945350647) tensor(928080849)
elements in the bin: 17269798
nonzeros: 16548
65
tensor(928080849) tensor(909264129)
elements in t

68
tensor(868373694) tensor(845004928)
elements in the bin: 23368766
nonzeros: 12067
69
tensor(845004928) tensor(819578770)
elements in the bin: 25426158
nonzeros: 12067
70
tensor(819578770) tensor(792166198)
elements in the bin: 27412572
nonzeros: 12067
71
tensor(792166198) tensor(761981069)
elements in the bin: 30185129
nonzeros: 12067
72
tensor(761981069) tensor(727910950)
elements in the bin: 34070119
nonzeros: 12067
73
tensor(727910950) tensor(689451894)
elements in the bin: 38459056
nonzeros: 12067
74
tensor(689451894) tensor(646050825)
elements in the bin: 43401069
nonzeros: 12067
75
tensor(646050825) tensor(594340961)
elements in the bin: 51709864
nonzeros: 12067
76
tensor(594340961) tensor(532250623)
elements in the bin: 62090338
nonzeros: 12067
77
tensor(532250623) tensor(451603387)
elements in the bin: 80647236
nonzeros: 12067
78
tensor(451603387) tensor(336357382)
elements in the bin: 115246005
nonzeros: 12067
79
tensor(336357382) tensor(1492938)
elements in the bin: 334864

3.94e+08
7.86e+08
1.57e+09
3.11e+09
6.12e+09
1.19e+10
2.25e+10
4.09e+10
6.99e+10
1.11e+11
1.64e+11
2.26e+11
2.95e+11
3.70e+11
4.48e+11
5.30e+11
6.14e+11
7.00e+11
381892 [381892. 381892. 381892. 381892. 381892. 381892. 381892.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0014)
0
tensor(1279999996) tensor(1279630902)
elements in the bin: 369094
nonzeros: 4226
1
tensor(1279630902) tensor(1279074315)
elements in the bin: 556587
nonzeros: 4226
2
tensor(1279074315) tensor(1278373312)
elements in the bin: 701003
nonzeros: 4226
3
tensor(1278373312) tensor(1277576215)
elements in the bin: 797097
nonzeros: 4226
4
tensor(1277576215) tensor(1276655089)
elements in the bin: 921126
nonzeros: 4226
5
tensor(1276655089) tensor(1275662721)
elements in the bin: 992368
nonzeros: 4226
6
tensor(1275662721) tensor(1274554041)
elements in the bin: 1108680
nonzeros: 4226
7
tensor(1274554041) tensor(1273340578)
elements in the bin: 1213463
nonzeros: 4226
8
tensor(1273340578) tensor(1272061255)
elements in t

30
tensor(1222335282) tensor(1218495175)
elements in the bin: 3840107
nonzeros: 2240
31
tensor(1218495175) tensor(1214689404)
elements in the bin: 3805771
nonzeros: 2240
32
tensor(1214689404) tensor(1210615327)
elements in the bin: 4074077
nonzeros: 2240
33
tensor(1210615327) tensor(1206344436)
elements in the bin: 4270891
nonzeros: 2240
34
tensor(1206344436) tensor(1201948289)
elements in the bin: 4396147
nonzeros: 2240
35
tensor(1201948289) tensor(1197578853)
elements in the bin: 4369436
nonzeros: 2240
36
tensor(1197578853) tensor(1193016007)
elements in the bin: 4562846
nonzeros: 2240
37
tensor(1193016007) tensor(1188196464)
elements in the bin: 4819543
nonzeros: 2240
38
tensor(1188196464) tensor(1183131533)
elements in the bin: 5064931
nonzeros: 2240
39
tensor(1183131533) tensor(1177771309)
elements in the bin: 5360224
nonzeros: 2240
40
tensor(1177771309) tensor(1172342518)
elements in the bin: 5428791
nonzeros: 2240
41
tensor(1172342518) tensor(1166546577)
elements in the bin: 579

47
tensor(1134419720) tensor(1126857534)
elements in the bin: 7562186
nonzeros: 1155
48
tensor(1126857534) tensor(1118784305)
elements in the bin: 8073229
nonzeros: 1155
49
tensor(1118784305) tensor(1110771854)
elements in the bin: 8012451
nonzeros: 1155
50
tensor(1110771854) tensor(1101903799)
elements in the bin: 8868055
nonzeros: 1155
51
tensor(1101903799) tensor(1093193048)
elements in the bin: 8710751
nonzeros: 1155
52
tensor(1093193048) tensor(1084334482)
elements in the bin: 8858566
nonzeros: 1155
53
tensor(1084334482) tensor(1074211823)
elements in the bin: 10122659
nonzeros: 1155
54
tensor(1074211823) tensor(1064282221)
elements in the bin: 9929602
nonzeros: 1155
55
tensor(1064282221) tensor(1053688180)
elements in the bin: 10594041
nonzeros: 1155
56
tensor(1053688180) tensor(1043164283)
elements in the bin: 10523897
nonzeros: 1155
57
tensor(1043164283) tensor(1031754725)
elements in the bin: 11409558
nonzeros: 1155
58
tensor(1031754725) tensor(1019497339)
elements in the bin:

64
tensor(949784815) tensor(931806362)
elements in the bin: 17978453
nonzeros: 587
65
tensor(931806362) tensor(913418592)
elements in the bin: 18387770
nonzeros: 587
66
tensor(913418592) tensor(895214732)
elements in the bin: 18203860
nonzeros: 587
67
tensor(895214732) tensor(874793156)
elements in the bin: 20421576
nonzeros: 587
68
tensor(874793156) tensor(853541420)
elements in the bin: 21251736
nonzeros: 587
69
tensor(853541420) tensor(829136203)
elements in the bin: 24405217
nonzeros: 587
70
tensor(829136203) tensor(801368049)
elements in the bin: 27768154
nonzeros: 587
71
tensor(801368049) tensor(770881206)
elements in the bin: 30486843
nonzeros: 587
72
tensor(770881206) tensor(738555401)
elements in the bin: 32325805
nonzeros: 587
73
tensor(738555401) tensor(701626715)
elements in the bin: 36928686
nonzeros: 587
74
tensor(701626715) tensor(660639769)
elements in the bin: 40986946
nonzeros: 587
75
tensor(660639769) tensor(609284488)
elements in the bin: 51355281
nonzeros: 587
76
t

## different amp_rounds_strange 

In [12]:
k = 0.1 * N_0

for amp_rounds in range(0,30,5):

    print('amplification_rounds = %d'%amp_rounds)
    
    data, counts, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0, k=k, amp_rounds=amp_rounds)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_strange_%damp_rounds.csv'%amp_rounds)
    
    del(log10p_vector)
    del(sorted_log10p_vector)

amplification_rounds = 0
12202649 [12202649. 12202649. 12202649. 12202649. 12202649. 12202649. 12202649.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0065)
0
tensor(1279999996) tensor(1279622767)
elements in the bin: 377229
nonzeros: 19192
1
tensor(1279622767) tensor(1279055760)
elements in the bin: 567007
nonzeros: 19192
2
tensor(1279055760) tensor(1278361488)
elements in the bin: 694272
nonzeros: 19192
3
tensor(1278361488) tensor(1277550945)
elements in the bin: 810543
nonzeros: 19192
4
tensor(1277550945) tensor(1276641191)
elements in the bin: 909754
nonzeros: 19192
5
tensor(1276641191) tensor(1275619059)
elements in the bin: 1022132
nonzeros: 19192
6
tensor(1275619059) tensor(1274504088)
elements in the bin: 1114971
nonzeros: 19192
7
tensor(1274504088) tensor(1273300941)
elements in the bin: 1203147
nonzeros: 19192
8
tensor(1273300941) tensor(1272003701)
elements in the bin: 1297240
nonzeros: 19192
9
tensor(1272003701) tensor(1270624203)
elements in the bin: 1379498
nonzeros: 1

32
tensor(1213859873) tensor(1209780729)
elements in the bin: 4079144
nonzeros: 19142
33
tensor(1209780729) tensor(1205539413)
elements in the bin: 4241316
nonzeros: 19142
34
tensor(1205539413) tensor(1201186010)
elements in the bin: 4353403
nonzeros: 19142
35
tensor(1201186010) tensor(1196634041)
elements in the bin: 4551969
nonzeros: 19142
36
tensor(1196634041) tensor(1192000860)
elements in the bin: 4633181
nonzeros: 19142
37
tensor(1192000860) tensor(1187126997)
elements in the bin: 4873863
nonzeros: 19142
38
tensor(1187126997) tensor(1181969161)
elements in the bin: 5157836
nonzeros: 19142
39
tensor(1181969161) tensor(1176620152)
elements in the bin: 5349009
nonzeros: 19142
40
tensor(1176620152) tensor(1171062271)
elements in the bin: 5557881
nonzeros: 19142
41
tensor(1171062271) tensor(1165307981)
elements in the bin: 5754290
nonzeros: 19142
42
tensor(1165307981) tensor(1159346983)
elements in the bin: 5960998
nonzeros: 19142
43
tensor(1159346983) tensor(1153048645)
elements in t

51
tensor(1101173842) tensor(1092436704)
elements in the bin: 8737138
nonzeros: 19042
52
tensor(1092436704) tensor(1083212394)
elements in the bin: 9224310
nonzeros: 19042
53
tensor(1083212394) tensor(1073575437)
elements in the bin: 9636957
nonzeros: 19042
54
tensor(1073575437) tensor(1063576633)
elements in the bin: 9998804
nonzeros: 19042
55
tensor(1063576633) tensor(1053079325)
elements in the bin: 10497308
nonzeros: 19042
56
tensor(1053079325) tensor(1041897504)
elements in the bin: 11181821
nonzeros: 19042
57
tensor(1041897504) tensor(1030245507)
elements in the bin: 11651997
nonzeros: 19042
58
tensor(1030245507) tensor(1018056099)
elements in the bin: 12189408
nonzeros: 19042
59
tensor(1018056099) tensor(1004960653)
elements in the bin: 13095446
nonzeros: 19042
60
tensor(1004960653) tensor(991166741)
elements in the bin: 13793912
nonzeros: 19042
61
tensor(991166741) tensor(976629163)
elements in the bin: 14537578
nonzeros: 19042
62
tensor(976629163) tensor(961453647)
elements in

67
tensor(889639391) tensor(868394130)
elements in the bin: 21245261
nonzeros: 18915
68
tensor(868394130) tensor(844930373)
elements in the bin: 23463757
nonzeros: 18915
69
tensor(844930373) tensor(819683235)
elements in the bin: 25247138
nonzeros: 18915
70
tensor(819683235) tensor(792122924)
elements in the bin: 27560311
nonzeros: 18915
71
tensor(792122924) tensor(761851677)
elements in the bin: 30271247
nonzeros: 18915
72
tensor(761851677) tensor(728154511)
elements in the bin: 33697166
nonzeros: 18915
73
tensor(728154511) tensor(689727808)
elements in the bin: 38426703
nonzeros: 18915
74
tensor(689727808) tensor(646064295)
elements in the bin: 43663513
nonzeros: 18915
75
tensor(646064295) tensor(594362127)
elements in the bin: 51702168
nonzeros: 18915
76
tensor(594362127) tensor(531540002)
elements in the bin: 62822125
nonzeros: 18915
77
tensor(531540002) tensor(450772906)
elements in the bin: 80767096
nonzeros: 18915
78
tensor(450772906) tensor(333818207)
elements in the bin: 11695

79
tensor(334002162) tensor(1128260)
elements in the bin: 332873902
nonzeros: 18816
amplification_rounds = 25
1.68e+06
1.82e+06
1.96e+06
2.11e+06
2.25e+06
2.40e+06
2.54e+06
2.68e+06
2.83e+06
2.98e+06
3.12e+06
3.27e+06
3.42e+06
3.56e+06
3.71e+06
3.86e+06
4.01e+06
4.16e+06
4.30e+06
4.45e+06
4.60e+06
4.75e+06
4.90e+06
5.05e+06
5.20e+06
12197930 [12197930. 12197930. 12197930. 12197930. 12197930. 12197930. 12197930.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0063)
0
tensor(1279999996) tensor(1279622489)
elements in the bin: 377507
nonzeros: 18654
1
tensor(1279622489) tensor(1279055790)
elements in the bin: 566699
nonzeros: 18654
2
tensor(1279055790) tensor(1278363479)
elements in the bin: 692311
nonzeros: 18654
3
tensor(1278363479) tensor(1277548142)
elements in the bin: 815337
nonzeros: 18654
4
tensor(1277548142) tensor(1276639227)
elements in the bin: 908915
nonzeros: 18654
5
tensor(1276639227) tensor(1275618054)
elements in the bin: 1021173
nonzeros: 18654
6
tensor(1275618054) tens

In [10]:
p_variable = np.array(p_variable)
np.save('p_variable_Byrne.npy', p_variable)

## different R

In [8]:
for t in [2, 4, 8, 16]:
    
    R = counts.sum()
    F = np.loadtxt('../binning_real_data_q/F_Byrne.csv')[0]
    N_0 = int(R / (F-1))
    R = int(R / t)
    
    
    print('R = %d'%R)
    
    data, counts, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_%dt.csv'%t)
    
    del(log10p_vector)
    del(sorted_log10p_vector)

R = 6097736
3.08e+06
6.16e+06
1.23e+07
2.47e+07
4.93e+07
9.86e+07
1.97e+08
3.94e+08
7.86e+08
1.57e+09
3.11e+09
6.12e+09
1.19e+10
2.25e+10
4.09e+10
6.99e+10
1.11e+11
1.64e+11
2.26e+11
2.95e+11
3.70e+11
4.48e+11
5.30e+11
6.14e+11
7.00e+11
6096836 [6096836. 6096836. 6096836. 6096836. 6096836. 6096836. 6096836.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0063)
0
tensor(1279999997) tensor(1279622269)
elements in the bin: 377728
nonzeros: 18829
1
tensor(1279622269) tensor(1279056147)
elements in the bin: 566122
nonzeros: 18829
2
tensor(1279056147) tensor(1278363660)
elements in the bin: 692487
nonzeros: 18829
3
tensor(1278363660) tensor(1277550121)
elements in the bin: 813539
nonzeros: 18829
4
tensor(1277550121) tensor(1276639727)
elements in the bin: 910394
nonzeros: 18829
5
tensor(1276639727) tensor(1275622222)
elements in the bin: 1017505
nonzeros: 18829
6
tensor(1275622222) tensor(1274508540)
elements in the bin: 1113682
nonzeros: 18829
7
tensor(1274508540) tensor(1273309578)
elemen

20
tensor(1251759913) tensor(1249333929)
elements in the bin: 2425984
nonzeros: 8282
21
tensor(1249333929) tensor(1246754640)
elements in the bin: 2579289
nonzeros: 8282
22
tensor(1246754640) tensor(1244061656)
elements in the bin: 2692984
nonzeros: 8282
23
tensor(1244061656) tensor(1241267218)
elements in the bin: 2794438
nonzeros: 8282
24
tensor(1241267218) tensor(1238346308)
elements in the bin: 2920910
nonzeros: 8282
25
tensor(1238346308) tensor(1235315952)
elements in the bin: 3030356
nonzeros: 8282
26
tensor(1235315952) tensor(1232153758)
elements in the bin: 3162194
nonzeros: 8282
27
tensor(1232153758) tensor(1228828712)
elements in the bin: 3325046
nonzeros: 8282
28
tensor(1228828712) tensor(1225444449)
elements in the bin: 3384263
nonzeros: 8282
29
tensor(1225444449) tensor(1221888958)
elements in the bin: 3555491
nonzeros: 8282
30
tensor(1221888958) tensor(1218163490)
elements in the bin: 3725468
nonzeros: 8282
31
tensor(1218163490) tensor(1214299505)
elements in the bin: 386

45
tensor(1146156902) tensor(1139627261)
elements in the bin: 6529641
nonzeros: 1512
46
tensor(1139627261) tensor(1132456006)
elements in the bin: 7171255
nonzeros: 1512
47
tensor(1132456006) tensor(1125072515)
elements in the bin: 7383491
nonzeros: 1512
48
tensor(1125072515) tensor(1117459079)
elements in the bin: 7613436
nonzeros: 1512
49
tensor(1117459079) tensor(1109404945)
elements in the bin: 8054134
nonzeros: 1512
50
tensor(1109404945) tensor(1101259766)
elements in the bin: 8145179
nonzeros: 1512
51
tensor(1101259766) tensor(1091997526)
elements in the bin: 9262240
nonzeros: 1512
52
tensor(1091997526) tensor(1082778333)
elements in the bin: 9219193
nonzeros: 1512
53
tensor(1082778333) tensor(1073351883)
elements in the bin: 9426450
nonzeros: 1512
54
tensor(1073351883) tensor(1063527638)
elements in the bin: 9824245
nonzeros: 1512
55
tensor(1063527638) tensor(1053025512)
elements in the bin: 10502126
nonzeros: 1512
56
tensor(1053025512) tensor(1041830005)
elements in the bin: 11

61
tensor(989139303) tensor(974679190)
elements in the bin: 14460113
nonzeros: 118
62
tensor(974679190) tensor(960667486)
elements in the bin: 14011704
nonzeros: 118
63
tensor(960667486) tensor(944199555)
elements in the bin: 16467931
nonzeros: 118
64
tensor(944199555) tensor(927230881)
elements in the bin: 16968674
nonzeros: 118
65
tensor(927230881) tensor(909317320)
elements in the bin: 17913561
nonzeros: 118
66
tensor(909317320) tensor(890454133)
elements in the bin: 18863187
nonzeros: 118
67
tensor(890454133) tensor(869806702)
elements in the bin: 20647431
nonzeros: 118
68
tensor(869806702) tensor(847954303)
elements in the bin: 21852399
nonzeros: 118
69
tensor(847954303) tensor(822127723)
elements in the bin: 25826580
nonzeros: 118
70
tensor(822127723) tensor(796584421)
elements in the bin: 25543302
nonzeros: 118
71
tensor(796584421) tensor(765312473)
elements in the bin: 31271948
nonzeros: 118
72
tensor(765312473) tensor(732098101)
elements in the bin: 33214372
nonzeros: 118
73
t

## different k

In [12]:
R = counts.sum()
F = np.loadtxt('../binning_real_data_q/F_Byrne.csv')[0]
N_0 = int(R / (F-1))

data, counts, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, N_0=N_0, k=1e9)
fields = infer_fields(data, counts)
log10p_vector = log10p_vector_from_fields(fields)
sorted_log10p_vector = log10p_vector.sort()[0]
log10q_vector = log10q_vector_func(data, fields)
argsort = np.argsort(log10q_vector)
sorted_log10q_vector = log10q_vector[argsort][::-1]
counts = counts[argsort][::-1]

bins=80
df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
df_bins.to_csv('df_bins_simulation_Byrne_smallk.csv')

del(log10p_vector)
del(sorted_log10p_vector)

p_variable = np.array(p_variable)
np.save('p_variable_Byrne_smallk.npy', p_variable)

3.85e+05
7.69e+05
1.54e+06
3.07e+06
6.14e+06
1.22e+07
2.43e+07
4.81e+07
9.39e+07
1.80e+08
3.32e+08
5.82e+08
9.49e+08
1.44e+09
2.03e+09
2.70e+09
3.42e+09
4.20e+09
5.01e+09
5.84e+09
6.69e+09
7.56e+09
8.45e+09
9.34e+09
1.02e+10
1525557 [1525557. 1525557. 1525557. 1525557. 1525557. 1525557. 1525557.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0008)
0
tensor(1279999992) tensor(1279619161)
elements in the bin: 380831
nonzeros: 2402
1
tensor(1279619161) tensor(1279055967)
elements in the bin: 563194
nonzeros: 2402
2
tensor(1279055967) tensor(1278371147)
elements in the bin: 684820
nonzeros: 2402
3
tensor(1278371147) tensor(1277565271)
elements in the bin: 805876
nonzeros: 2402
4
tensor(1277565271) tensor(1276652766)
elements in the bin: 912505
nonzeros: 2402
5
tensor(1276652766) tensor(1275652997)
elements in the bin: 999769
nonzeros: 2402
6
tensor(1275652997) tensor(1274526607)
elements in the bin: 1126390
nonzeros: 2402
7
tensor(1274526607) tensor(1273312774)
elements in the bin: 12138

In [13]:
p_variable

array([0.99980771, 0.99961553, 0.99923151, 0.99846479, 0.99693658,
       0.99390121, 0.98791285, 0.97625535, 0.95413675, 0.91413504,
       0.8476059 , 0.75063962, 0.63228683, 0.51300949, 0.41046602,
       0.33049108, 0.27061412, 0.22600596, 0.19235699, 0.16649181,
       0.146202  , 0.12997711, 0.11677261, 0.10585486, 0.09670257])

## different k (a lot of different k)

In [10]:
for k in [1e1, 1e3]:

    #print('amplification_rounds = %d'%amp_rounds)
    
    data, counts, p_variable = create_simulated_dataset(p_vector= 10**(data_log10p_vector), R=R, k=k, N_0=N_0)
    fields = infer_fields(data, counts)
    log10p_vector = log10p_vector_from_fields(fields)
    sorted_log10p_vector = log10p_vector.sort()[0]
    log10q_vector = log10q_vector_func(data, fields)
    argsort = np.argsort(log10q_vector)
    sorted_log10q_vector = log10q_vector[argsort][::-1]
    counts = counts[argsort][::-1]
    
    bins=80
    df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins)#, writefolder='results_ByrneT0_IM', step=1)
    df_bins.to_csv('df_bins_simulation_Byrne_%.1ek.csv'%k)
    
    np.save('p_simulation_Byrne_%.1ek.npy'%k,p_variable)
    
    del(log10p_vector)
    del(sorted_log10p_vector)

1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
1.54e+06
12196031 [12196031. 12196031. 12196031. 12196031. 12196031. 12196031. 12196031.]
0
1
2
3
4
5
6
tensor(1.)
0
1
2
3
4
5
6
tensor(0.0065)
0
tensor(1279999996) tensor(1279622722)
elements in the bin: 377274
nonzeros: 19192
1
tensor(1279622722) tensor(1279055497)
elements in the bin: 567225
nonzeros: 19192
2
tensor(1279055497) tensor(1278362717)
elements in the bin: 692780
nonzeros: 19192
3
tensor(1278362717) tensor(1277550520)
elements in the bin: 812197
nonzeros: 19192
4
tensor(1277550520) tensor(1276638955)
elements in the bin: 911565
nonzeros: 19192
5
tensor(1276638955) tensor(1275617032)
elements in the bin: 1021923
nonzeros: 19192
6
tensor(1275617032) tensor(1274504768)
elements in the bin: 1112264
nonzeros: 19192
7
tensor(1274504768) tensor(1273299496)
elements i

36
tensor(1196646408) tensor(1192003530)
elements in the bin: 4642878
nonzeros: 19192
37
tensor(1192003530) tensor(1187094817)
elements in the bin: 4908713
nonzeros: 19192
38
tensor(1187094817) tensor(1181954367)
elements in the bin: 5140450
nonzeros: 19192
39
tensor(1181954367) tensor(1176600938)
elements in the bin: 5353429
nonzeros: 19192
40
tensor(1176600938) tensor(1171043427)
elements in the bin: 5557511
nonzeros: 19192
41
tensor(1171043427) tensor(1165310995)
elements in the bin: 5732432
nonzeros: 19192
42
tensor(1165310995) tensor(1159323622)
elements in the bin: 5987373
nonzeros: 19192
43
tensor(1159323622) tensor(1152986291)
elements in the bin: 6337331
nonzeros: 19192
44
tensor(1152986291) tensor(1146543660)
elements in the bin: 6442631
nonzeros: 19192
45
tensor(1146543660) tensor(1139747400)
elements in the bin: 6796260
nonzeros: 19192
46
tensor(1139747400) tensor(1132672045)
elements in the bin: 7075355
nonzeros: 19192
47
tensor(1132672045) tensor(1125320635)
elements in t